# 06 · Demo (Fase 6)
Prueba la lógica de inferencia de la demo (`src/inspectia/demo.py`) **sin interfaz** y deja armada la app Gradio.

- Usa la CNN oficial (`cnn_resnet18.pt`, semilla 42) con el umbral fijado en **validación**. No toca el test.
- Verifica que el pipeline de la demo (imagen suelta → probabilidad) reproduce los scores guardados de validación.
- Todo lo de la interfaz está en `demo/app.py`; el Space de Hugging Face se arma con `scripts/build_space.py`.

In [8]:
# --- Bootstrap: same cell at the top of every notebook ---
import os
import subprocess
import sys
from pathlib import Path

REPO = "matifleman/inspectia"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/InspectIA")
    REPO_DIR = Path("/content/inspectia")
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", f"https://github.com/{REPO}.git", str(REPO_DIR)], check=True)
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
else:
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DRIVE_ROOT = REPO_DIR
    if "WANDB_API_KEY" not in os.environ:
        os.environ["WANDB_MODE"] = "offline"  # local runs without a key log offline

os.chdir(REPO_DIR)
if IN_COLAB:  # locally, install once in your venv (see README)
    %pip install -q -r requirements.txt -e .
# Editable installs are only picked up after a kernel restart; this makes the
# package importable in the current session too.
sys.path.insert(0, str(REPO_DIR / "src"))

DATA_DIR = DRIVE_ROOT / "data"
CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints"
SPLITS_DIR = DRIVE_ROOT / "splits"

from inspectia.utils import get_device, set_seed

SEED = 42
set_seed(SEED)
device = get_device()
print(f"Repo: {REPO_DIR} | Data: {DATA_DIR} | Device: {device}")

Repo: /Users/ipejinakis/Documents/Ingenieria-UCA/Inteligencia Artificial/InspectIA/repo | Data: /Users/ipejinakis/Documents/Ingenieria-UCA/Inteligencia Artificial/InspectIA/repo/data | Device: mps


In [9]:
# Paths shared by every notebook after the bootstrap
DATASET_DIR = DATA_DIR / "KolektorSDD2"
SPLITS_DIR = REPO_DIR / "splits"            # versioned in git: the whole team uses the SAME splits
RESULTS_DIR = DRIVE_ROOT / "results"
FEATURES_DIR = DRIVE_ROOT / "features"
for d in (SPLITS_DIR, RESULTS_DIR, FEATURES_DIR, CHECKPOINT_DIR):
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from inspectia.demo import REVIEW_LOW, Inspector

insp = Inspector.from_artifacts(CHECKPOINT_DIR, RESULTS_DIR)
print(f"Umbral oficial (val, máx. F2): {insp.threshold:.4f} | zona REVISAR desde {REVIEW_LOW} | device: {insp.device}")
print("Métricas de validación:", {k: round(v, 3) for k, v in insp.val_metrics.items() if k in ('f2', 'recall', 'precision', 'avg_precision')})

In [ ]:
# Consistencia: las probabilidades de la demo deben coincidir con los scores de validación guardados (Fase 3).
val = pd.read_csv(SPLITS_DIR / "val.csv")
saved = np.load(RESULTS_DIR / "cnn_resnet18_val_scores.npz")
idx = list(range(0, 20)) + list(val.index[val.label == 1][:10])           # 20 primeras + 10 defectuosas
probs = np.array([insp.inspect(Image.open(DATASET_DIR / val.image[i])).prob for i in idx])
diff = np.abs(probs - saved["scores"][idx])
print(f"max |diff| = {diff.max():.2e} sobre {len(idx)} imágenes (esperado < 1e-3; difiere solo por CPU vs MPS/GPU)")
assert diff.max() < 1e-3, "El pipeline de la demo NO reproduce los scores de validación: revisar preprocesamiento"

In [ ]:
# Una buena y una defectuosa de VALIDACIÓN (nunca del test) con su Grad-CAM.
rows = [val[val.label == 0].iloc[0], val[val.label == 1].iloc[0]]
fig, axes = plt.subplots(2, 2, figsize=(7, 10))
for r, (ax_img, ax_cam) in zip(rows, axes):
    img = Image.open(DATASET_DIR / r.image)
    t0 = time.time(); res = insp.inspect(img); dt = time.time() - t0
    ax_img.imshow(img, cmap="gray"); ax_img.set_title(f"{r.image} (real: {'defectuosa' if r.label else 'buena'})", fontsize=8)
    ax_cam.imshow(res.overlay); ax_cam.set_title(f"{res.verdict} · p={res.prob:.3f} · {dt*1000:.0f} ms", fontsize=8)
    for a in (ax_img, ax_cam): a.axis("off")
plt.tight_layout(); plt.show()

## Lanzar la app y armar el Space
Local (desde la raíz del repo, con `pip install gradio` en el venv):
```bash
python demo/app.py
```
Space de Hugging Face:
```bash
python scripts/build_space.py --out space   # copia app, paquete, pesos y umbral (sin imágenes del dataset)
```
y subir el contenido de `space/` a un Space con SDK Gradio.